# Hands-On Lab — Cross-Validation
**Week 4 — Day 2 | BinX Tech AI & ML Internship**

---

## Objective

In the learning notebook I understood why a single validation split
can be unreliable and how k-fold cross-validation produces a more
stable performance estimate.

In this lab I apply the full CV workflow on the Student Performance
dataset — the same dataset and model from Day 1 — so I can make a
direct, meaningful comparison between the two approaches.

1. Create a train/test split and lock the test set immediately
2. Run 5-fold stratified cross-validation on the training portion
3. Report mean ± std and inspect individual fold scores
4. Confirm that stratification preserved class balance across folds
5. Compare the CV estimate to the Day 1 single-split score
6. Evaluate the final model once on the held-out test set

**Dataset:** Student Performance (2,392 students, 5-class grade prediction)  
**Model:** Random Forest — same as Day 1, so the comparison is fair

---

## Step 1 — Import Libraries

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, accuracy_score

SEED = 42

## Step 2 — Load the Data

Same dataset as Day 1 — 2,392 students, 5-class target (`GradeClass`).
I drop `StudentID` because it's an identifier, not a feature.
I use weighted F1 throughout because the class sizes are unequal.

In [2]:
df = pd.read_csv(
    "../../Week1/mini_project/Data/Student_performance_data _.csv"
)

# StudentID is an identifier, not a useful feature
df = df.drop(columns=["StudentID"])


# Separate features (X) from the target (y)
X = df.drop(columns=["GradeClass"])

y = df["GradeClass"].astype(int)


# Check dataset size
print(
    f"Shape: {df.shape[0]} rows, "
    f"{X.shape[1]} features"
)


# Check the number of samples in each class
print("\nClass distribution:")
print(
    y.value_counts().sort_index()
)


# Check the proportion of Class 4
print(
    f"\nClass 4 (grade F) proportion: "
    f"{(y == 4).mean():.1%}"
)

Shape: 2392 rows, 13 features

Class distribution:
GradeClass
0     107
1     269
2     391
3     414
4    1211
Name: count, dtype: int64

Class 4 (grade F) proportion: 50.6%


## Step 3 — Create the Train / Test Split

In Day 2, I only need one fixed split — train and test.
The CV folds handle all validation internally within the training
portion, so I pass the full 80% to cross-validation instead
of carving out a fixed validation slice.

The test set is locked immediately and stays untouched.

In [3]:
# Create one fixed train/test split
# 80% will be used for training and cross-validation
# 20% will remain locked for the final evaluation

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    
    # Keep 20% for the final test set
    test_size=0.2,
    
    # Make the split reproducible
    random_state=SEED,
    
    # Preserve class proportions in both sets
    stratify=y
)


# Check the number of samples in each portion
print(
    f"Training portion: {X_train.shape[0]} rows => CV runs here"
)

print(
    f"Test set:    {X_test.shape[0]} rows  => locked"
)

print()


# Check whether Class 4 has a similar proportion
# in both the training and test sets

print(
    f"Class 4 in train: {(y_train == 4).mean():.3f}"
)

print(
    f"Class 4 in test:  {(y_test == 4).mean():.3f}"
)

Training portion: 1913 rows => CV runs here
Test set:    479 rows  => locked

Class 4 in train: 0.506
Class 4 in test:  0.507


## Step 4 — Set Up Stratified 5-Fold CV

I create `StratifiedKFold` explicitly rather than relying on the
default — this makes the stratification visible and intentional.

With 5 unequal classes (Class 4 at 50.6%), stratification is
especially important here. A fold that accidentally gets too many
or too few Class 4 students will produce a weighted F1 that doesn't
reflect the true difficulty of the task.

In [4]:
skf = StratifiedKFold(
    n_splits=5,        # 5 rotating folds
    shuffle=True,      # randomize before splitting
    random_state=SEED  # reproducible
)

## Step 5 — Run cross_val_score

Same Random Forest as Day 1 (`max_depth=None`, 100 trees) —
keeping the model identical makes the Day 1 vs Day 2 comparison fair.

In [5]:
model = RandomForestClassifier(
    max_depth=None,     # same setting chosen in Day 1
    n_estimators=100,
    random_state=SEED
)

# Run 5-fold stratified CV on the training portion
scores = cross_val_score(
    model,
    X_train, y_train,
    cv=skf,
    scoring="f1_weighted"  # weighted because classes are imbalanced
)

# One score per fold
print("Weighted F1 per fold:")
for i, s in enumerate(scores, 1):
    print(f"  Fold {i}: {s:.3f}")

Weighted F1 per fold:
  Fold 1: 0.913
  Fold 2: 0.924
  Fold 3: 0.927
  Fold 4: 0.890
  Fold 5: 0.930


The scores vary slightly across folds — this is expected. Each fold
contains different students, so the exact mix of easy and hard cases
shifts. Fold 4 is the weakest at 0.890; Fold 5 is the strongest at 0.930.
That range of 0.040 is the variation a single validation split could
have hidden completely.

## Step 6 — Mean and Standard Deviation

In [6]:
print(f"Mean F1: {scores.mean():.3f}")
print(f"Std F1:  {scores.std():.3f}")
print()
print(f"CV estimate: {scores.mean():.3f} ± {scores.std():.3f}")

Mean F1: 0.917
Std F1:  0.014

CV estimate: 0.917 ± 0.014


The std of 0.014 is very low — the model performs consistently
across all five folds. With 1,913 training rows, each fold has
enough samples to be a stable representative of the full dataset.
The mean of 0.917 is the number to rely on for development decisions.

## Step 7 — Confirm Stratified Folds

I'll inspect the class proportions in each fold's validation set
to confirm that stratification kept Class 4 near its original 50.6%.

In [7]:
print(
    f"Original Class 4 proportion: "
    f"{(y_train == 4).mean():.3f}\n"
)


print(
    f"{'Fold':<6}"
    f"{'Val rows':<10}"
    f"{'Class 4 prop':<14}"
    f"{'Fold F1'}"
)

print("-" * 44)


for fold_num, (tr_idx, val_idx) in enumerate(
    skf.split(X_train, y_train),
    1
):
    
    fold_y = y_train.iloc[val_idx]
    
    class4 = (fold_y == 4).mean()
    
    fold_f1 = scores[fold_num - 1]
    
    print(
        f"{fold_num:<6}"
        f"{len(val_idx):<10}"
        f"{class4:<14.3f}"
        f"{fold_f1:.3f}"
    )

Original Class 4 proportion: 0.506

Fold  Val rows  Class 4 prop  Fold F1
--------------------------------------------
1     383       0.504         0.913
2     383       0.507         0.924
3     383       0.507         0.927
4     382       0.508         0.890
5     382       0.505         0.930


Class 4 proportion stays at ~0.51 across every fold — stratification
is working correctly. Without it, some folds might have had 60%+ of
Class 4 samples and others far less, making fold-to-fold comparisons
unreliable.

## Step 8 — Compare with Day 1

In Day 1, I used a fixed validation split of 479 rows and got:

```text
Day 1 single-split Val F1: 0.904
```

Cross-validation across 5 folds gives:

```text
Day 2 CV Mean F1: 0.917 ± 0.014
```

The CV estimate is slightly higher (+0.013) — but the more important
point is the std of 0.014. It tells me that performance is stable:
no matter which slice of data ends up as validation, the model
scores somewhere between ~0.903 and ~0.931.

Unlike the Pima dataset in the learning notebook (where CV revealed
that Day 1's single split was optimistically off by 0.081), here
the two estimates are close. That's because 2,392 rows is large
enough that even a single random split tends to be representative.
Cross-validation still provides the more complete picture — five
independent estimates instead of one.

## Step 9 — Final Evaluation on the Test Set 

The CV estimate is for development. Now I train the final model
on the full training portion and open the test set exactly once.

In [8]:
# Train the final model on the full training portion

final_model = RandomForestClassifier(
    max_depth=None,
    n_estimators=100,
    random_state=SEED
)

final_model.fit(
    X_train,
    y_train
)


# Make predictions on the locked test set

y_test_pred = final_model.predict(X_test)


# Evaluate the final model

test_f1 = f1_score(
    y_test,
    y_test_pred,
    average="weighted"
)

test_acc = accuracy_score(
    y_test,
    y_test_pred
)


print(
    f"CV estimate:    "
    f"{scores.mean():.3f} ± {scores.std():.3f}  (development)"
)

print(
    f"Test F1:        "
    f"{test_f1:.3f}                    (final honest estimate)"
)

print(
    f"Test Accuracy:  "
    f"{test_acc:.3f}"
)

CV estimate:    0.917 ± 0.014  (development)
Test F1:        0.905                    (final honest estimate)
Test Accuracy:  0.912


The test F1 (0.905) sits just inside the CV range — a healthy sign
that the CV estimate was an accurate prediction of real generalization.
The test set had no influence on any decision, so this number is honest.

## Summary

| Step | What I Did | Result |
|------|-----------|--------|
| Split | 80/20 stratified — test locked | 1913 / 479 rows |
| CV setup | StratifiedKFold, 5 folds, explicit | Class 4 ~0.51 per fold  |
| CV scores | cross_val_score, weighted F1 | 0.913 / 0.924 / 0.927 / 0.890 / 0.930 |
| CV estimate | Mean ± Std | **0.917 ± 0.014** |
| Day 1 comparison | Single split vs CV | 0.904 → 0.917 (+0.013) |
| Final test | Opened once, after all decisions | **Test F1 = 0.905**  |

**What cross-validation added over Day 1:**
Instead of one score from one slice of data, I now have five independent
estimates. The std of 0.014 tells me the model is stable — the Day 1
score of 0.904 wasn't a lucky outlier, it was a fair reflection of
consistent performance.

**What cross-validation did not do:**
It did not replace the test set. The test set answered a different
question — how the finalized model performs on data it never saw
in any form. CV and the test set are complementary, not interchangeable.